# Stoa 01 — download raw data

Download-only version of `download.py`. Every source is saved untouched in `data/raw/`, with a `manifest.json` entry (URL, SHA-256, size, download time, licence) per file.

Sources are downloaded in this order:

| # | Source (`SOURCES` name) | Saved as | Coverage (checked 2026-10-20) |
|---|---|---|---|
| 1 | PCS archive (`pcs_archive`) | `pcs_archive_2009_2022.csv.zip` | 2009-05-01 → 2022-10-31 |
| 2 | PCS Wayback copy (`pcs_wayback`) | `pcs_counts_wayback_20250502183548.csv` | 2021-07-01 → 2025-05-02 |
| 3 | PCS live (`pcs_live`) | `pcs_live_counts_<date>.parquet` | rolling ~2 years (2024-10-21 → today) |
| 4 | Sensor locations (`sensor_locations`) | `pcs_sensor_locations.parquet` | current snapshot |
| 5 | Sensor locations 2022 (`sensor_locations_2022`) | `pcs_sensor_locations_2022-10-24_wayback.json` | snapshot incl. 18 retired sensors |
| 6 | Footpath steepness (`footpath_steepness`) | `footpath_steepness.geojson` | static snapshot |
| 7 | GA 5 m LiDAR heights (`ga_dem`) | `ga_dem_5m_samples.json` | ground height at 5 points around every sensor (static); needs 4 and 5 |
| 8 | ERA5 weather (`weather`) | `open_meteo_era5/era5_hourly_<year>.json` | 2009-04-30 → ~6 days ago (UTC) |

Weather is limited to the four variables used downstream (`temperature_2m`, `precipitation`, `wind_speed_10m`, `is_day`). Microclimate is not downloaded.

This is the only notebook that goes online. Next: `02_combine.ipynb` reads everything, including the LiDAR heights, from `data/raw/`.

Keep every dated live file: the portal drops data older than ~2 years, so once its window passes 2025-05-02 (around May 2027) the old pulls are the only copy of those months.

## 1. Setup

In [ ]:
%pip install -q pyproj

In [ ]:
import os
import sys
from pathlib import Path

# Where the data lives. Colab: your Google Drive. A local clone of the repo: data/notebooks/,
# kept apart from the team pipeline's data/raw, data/interim and data/processed (git-ignored).
# Set STOA_NOTEBOOK_DATA to use another folder.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/stoa/data")
else:
    DATA = Path(os.environ.get("STOA_NOTEBOOK_DATA", Path.cwd().resolve().parent / "data" / "notebooks"))
DATA.mkdir(parents=True, exist_ok=True)
print("data folder:", DATA)

In [ ]:
import datetime as dt
import hashlib
import json
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from pyproj import Transformer

RAW = DATA / "raw"
MANIFEST = RAW / "manifest.json"

CKAN = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets"
CC_BY = "CC BY 4.0 (City of Melbourne Open Data)"
TODAY = dt.date.today()

# One point in central Melbourne (Town Hall); ERA5 snaps it to its 0.25 deg cell.
MELB_LAT, MELB_LON = -37.8136, 144.9631
WEATHER_VARS = [
    "temperature_2m", "precipitation", "wind_speed_10m", "is_day",
]
# Archive starts 2009-05-01 00:00 Melbourne time = 2009-04-30 14:00 UTC.
WEATHER_START = dt.date(2009, 4, 30)

## 2. Download helpers

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def stream(url: str, dest: Path, params: dict | None = None, retries: int = 4) -> None:
    tmp = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(retries):
        try:
            with requests.get(url, params=params, stream=True, timeout=(30, 900)) as r:
                r.raise_for_status()
                with tmp.open("wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            tmp.replace(dest)
            return
        except requests.RequestException as e:
            wait = 15 * (attempt + 1)
            print(f"  {dest.name}: {e} - retry in {wait}s", file=sys.stderr)
            time.sleep(wait)
    raise RuntimeError(f"failed to download {url}")


def record(manifest: dict, path: Path, url: str, licence: str, **extra) -> None:
    manifest[path.name] = {
        "url": url,
        "sha256": sha256(path),
        "bytes": path.stat().st_size,
        "downloaded_at_utc": dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
        "licence": licence,
        **extra,
    }
    print(f"  saved {path.name} ({path.stat().st_size / 1e6:.1f} MB)")


def portal_export(manifest: dict, dataset: str, fmt: str, dest: Path) -> None:
    url = f"{CKAN}/{dataset}/exports/{fmt}"
    print(f"{dataset} -> {dest.name}")
    stream(url, dest)
    meta = requests.get(f"{CKAN}/{dataset}", timeout=60).json()["metas"]["default"]
    record(manifest, dest, url, CC_BY, portal_records_count=meta.get("records_count"),
           portal_data_processed=meta.get("data_processed"))

## 3. Sources

In [ ]:
def get_pcs_live(m):
    portal_export(m, "pedestrian-counting-system-monthly-counts-per-hour", "parquet",
                  RAW / f"pcs_live_counts_{TODAY:%Y%m%d}.parquet")


def get_pcs_archive(m):
    ds = "pedestrian-counting-system-monthly-counts-per-hour"
    att = "pedestrian_counting_system_monthly_counts_per_hour_may_2009_to_14_dec_2022_csv_zip"
    url = f"{CKAN}/{ds}/attachments/{att}"
    dest = RAW / "pcs_archive_2009_2022.csv.zip"
    print(f"archive -> {dest.name}")
    stream(url, dest)
    record(m, dest, url, CC_BY)


def get_sensor_locations(m):
    portal_export(m, "pedestrian-counting-system-sensor-locations", "parquet",
                  RAW / "pcs_sensor_locations.parquet")


def get_sensor_locations_2022(m):
    # The current table omits 18 retired archive sensors. The Oct 2022 Socrata
    # copy of the same City of Melbourne table (status R/I/A) still lists them.
    src = "https://data.melbourne.vic.gov.au/resource/h57g-5234"
    url = f"http://web.archive.org/web/20221024034400id_/{src}"
    dest = RAW / "pcs_sensor_locations_2022-10-24_wayback.json"
    print(f"wayback sensor locations -> {dest.name}")
    stream(url, dest)
    record(m, dest, url, CC_BY, original_url=src, snapshot="2022-10-24T03:44:00Z")


PCS_WAYBACK_SNAPSHOTS = ["20250502183548"]


def get_pcs_wayback(m):
    # The live window rolls forward, so Nov 2022 - Oct 2024 is no longer on the
    # portal. Wayback holds full portal CSV exports made while it still was.
    # 20250502183548 covers 2021-07-01..2025-05-02 and matches the archive and
    # live rows it overlaps on ~98.5% of hours (rest differ by a few people).
    src = ("https://data.melbourne.vic.gov.au/explore/dataset/"
           "pedestrian-counting-system-monthly-counts-per-hour/download?format=csv")
    for ts in PCS_WAYBACK_SNAPSHOTS:
        url = f"http://web.archive.org/web/{ts}id_/{src}"
        dest = RAW / f"pcs_counts_wayback_{ts}.csv"
        print(f"wayback counts {ts} -> {dest.name}")
        stream(url, dest)
        record(m, dest, url, CC_BY, original_url=src, snapshot=ts)


def get_footpath_steepness(m):
    # GeoJSON keeps the footpath polygons needed for the point-in-section match.
    portal_export(m, "footpath-steepness", "geojson", RAW / "footpath_steepness.geojson")


def get_weather(m):
    url = "https://archive-api.open-meteo.com/v1/archive"
    out_dir = RAW / "open_meteo_era5"
    out_dir.mkdir(exist_ok=True)
    for year in range(WEATHER_START.year, TODAY.year + 1):
        start = max(dt.date(year, 1, 1), WEATHER_START)
        end = min(dt.date(year, 12, 31), TODAY)
        dest = out_dir / f"era5_hourly_{year}.json"
        params = {
            "latitude": MELB_LAT, "longitude": MELB_LON,
            "start_date": start.isoformat(), "end_date": end.isoformat(),
            "hourly": ",".join(WEATHER_VARS), "models": "era5",
            "timezone": "GMT", "wind_speed_unit": "kmh",
        }
        print(f"open-meteo {year} -> {dest.name}")
        for attempt in range(6):
            r = requests.get(url, params=params, timeout=(30, 300))
            if r.status_code == 429:
                time.sleep(65)
                continue
            r.raise_for_status()
            break
        else:
            raise RuntimeError(f"open-meteo kept rate limiting for {year}")
        dest.write_bytes(r.content)
        record(m, dest, r.url, "CC BY 4.0 (Open-Meteo; ERA5 Copernicus/ECMWF)",
               hours=len(r.json()["hourly"]["time"]))
        time.sleep(8)  # stay well under the per-minute weighted call limit


GA_DEM_IDENTIFY = "https://services.ga.gov.au/gis/rest/services/DEM_LiDAR_5m_2025/MapServer/identify"
DEM_STEP_M = 10.0


def get_ga_dem(m):
    # Ground heights from Geoscience Australia's 5 m LiDAR elevation model (a laser survey flown
    # over the city). Combine uses them only for sensors too far from any footpath section to take
    # slope/elevation from one, but which sensors those are is only known after its footpath match.
    # So every sensor is sampled here: at the sensor and 10 m east, west, north and south (combine
    # turns those into a slope). Points already in the file are not fetched again.
    dest = RAW / "ga_dem_5m_samples.json"
    print(f"GA LiDAR heights -> {dest.name}")
    cache = json.loads(dest.read_text()) if dest.exists() else {}

    # Sensor coordinates from the two sensor tables, exactly as combine reads them.
    cur = pd.read_parquet(RAW / "pcs_sensor_locations.parquet", columns=["location_id", "latitude", "longitude"])
    old = pd.DataFrame(json.loads((RAW / "pcs_sensor_locations_2022-10-24_wayback.json").read_text("utf-8")))
    old = pd.DataFrame({"location_id": old.sensor_id.astype(int),
                        "latitude": old.latitude.astype(float), "longitude": old.longitude.astype(float)})
    s = pd.concat([cur, old[~old.location_id.isin(cur.location_id)]], ignore_index=True).dropna()

    # Same steps as combine's fill_from_dem so the saved keys match exactly:
    # degrees -> metres (MGA zone 55), step 10 m, back to degrees, key rounded to 7 decimals.
    to_mga = Transformer.from_crs("EPSG:4326", "EPSG:7855", always_xy=True)
    to_wgs = Transformer.from_crs("EPSG:7855", "EPSG:4326", always_xy=True)
    xs, ys = to_mga.transform(s.longitude.to_numpy(), s.latitude.to_numpy())
    d = DEM_STEP_M
    points = {}
    for x, y in zip(xs, ys):
        for dx, dy in [(0, 0), (d, 0), (-d, 0), (0, d), (0, -d)]:
            lon, lat = to_wgs.transform(x + dx, y + dy)
            points[f"{lon:.7f},{lat:.7f}"] = (lon, lat)
    todo = [k for k in points if k not in cache]
    print(f"  {len(s)} sensors, {len(points)} points, {len(todo)} to fetch (~{len(todo) * 0.5 / 60:.0f} min)")

    with requests.Session() as session:
        for n, key in enumerate(todo, 1):
            lon, lat = points[key]
            params = {
                "geometry": f"{lon},{lat}", "geometryType": "esriGeometryPoint", "sr": 4326, "tolerance": 0,
                "mapExtent": f"{lon - .01},{lat - .01},{lon + .01},{lat + .01}", "imageDisplay": "400,400,96",
                "layers": "all", "returnGeometry": "false", "f": "json",
            }
            for attempt in range(4):
                try:
                    r = session.get(GA_DEM_IDENTIFY, params=params, timeout=60,
                                    headers={"User-Agent": "Mozilla/5.0 (pcs-preprocessing)"})
                    r.raise_for_status()
                    value = r.json()["results"][0]["attributes"]["Stretch.Pixel Value"]
                    # "NoData" = no ground surface in the model at that point (e.g. over the river
                    # at Sandridge Bridge). Saved as null; combine only fails if it actually needs it.
                    cache[key] = None if value == "NoData" else float(value)
                    break
                except (requests.RequestException, KeyError, IndexError, ValueError):
                    time.sleep(5 * (attempt + 1))
            else:
                raise RuntimeError(f"GA elevation service kept failing at {key}")
            if n % 100 == 0:   # save progress so an interrupted run can resume
                dest.write_text(json.dumps(cache, indent=2))
                print(f"  {n}/{len(todo)}")
    dest.write_text(json.dumps(cache, indent=2))
    print(f"  points with no LiDAR height (NoData): {sum(v is None for v in cache.values())}")
    record(m, dest, GA_DEM_IDENTIFY, "CC BY 4.0 (Geoscience Australia)", points=len(cache))


# Download order = the order the data is used: counts (oldest first), the sensor tables,
# footpath steepness, LiDAR heights (needs the sensor tables), then weather.
SOURCES = {
    "pcs_archive": get_pcs_archive,            # counts 2009-05 -> 2022-10
    "pcs_wayback": get_pcs_wayback,            # counts 2021-07 -> 2025-05
    "pcs_live": get_pcs_live,                  # counts, last ~2 years
    "sensor_locations": get_sensor_locations,  # current sensor table
    "sensor_locations_2022": get_sensor_locations_2022,  # retired sensors
    "footpath_steepness": get_footpath_steepness,
    "ga_dem": get_ga_dem,                      # needs both sensor tables
    "weather": get_weather,
}

## 4. Download

Takes a while: the weather step waits ~8 s between years to stay under Open-Meteo's rate limit, and the first LiDAR run asks for ~765 points at ~0.5 s each (~6-7 min; re-runs skip points already saved). A failed source keeps its last good copy and is reported at the end.

In [ ]:
# Set to a list of names from SOURCES to fetch only those, e.g. ["weather"].
ONLY = None

RAW.mkdir(parents=True, exist_ok=True)
manifest = json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}
failed = []
for name in ONLY or SOURCES:
    try:
        SOURCES[name](manifest)
    except Exception as e:  # noqa: BLE001 - one failed source is reported, the rest still download
        print(f"!! {name} failed: {e}", file=sys.stderr)
        failed.append(name)
    MANIFEST.write_text(json.dumps(manifest, indent=2))
if failed:
    raise RuntimeError(f"sources failed: {failed}")

## 5. Check coverage

Reads back what was downloaded and prints each source's actual first and last date, so gaps show up before the build step.

In [ ]:
import calendar
import zipfile


def span(name, dates, n=None):
    print(f"{name:<22} {dates.min():%Y-%m-%d} -> {dates.max():%Y-%m-%d}   rows {n if n is not None else len(dates):,}")


with zipfile.ZipFile(RAW / "pcs_archive_2009_2022.csv.zip") as z:
    csv = next(n for n in z.namelist() if n.endswith(".csv") and not n.startswith("__MACOSX"))
    a = pd.read_csv(z.open(csv), usecols=["Year", "Month", "Mdate"])
month = a.Month.map({m: i for i, m in enumerate(calendar.month_name) if m})
span("archive", pd.to_datetime(pd.DataFrame({"year": a.Year, "month": month, "day": a.Mdate})))

for p in sorted(RAW.glob("pcs_counts_wayback_*.csv")):
    w = pd.read_csv(p, sep=";", usecols=["sensing_date"])
    span(p.stem.replace("pcs_counts_", ""), pd.to_datetime(w.sensing_date))

for p in sorted(RAW.glob("pcs_live_counts_*.parquet")):
    lv = pd.read_parquet(p, columns=["sensing_date"])
    span(p.stem.replace("pcs_", ""), pd.to_datetime(lv.sensing_date))

hours = pd.concat([pd.DataFrame(json.loads(p.read_text())["hourly"])
                   for p in sorted((RAW / "open_meteo_era5").glob("era5_hourly_*.json"))])
hours = hours.dropna(subset=["temperature_2m"])
span("weather (UTC, non-null)", pd.to_datetime(hours.time))

print(f"{'sensor locations':<22} {len(pd.read_parquet(RAW / 'pcs_sensor_locations.parquet'))} sensors")
print(f"{'sensor locations 2022':<22} {len(json.loads((RAW / 'pcs_sensor_locations_2022-10-24_wayback.json').read_text()))} sensors")
print(f"{'footpath steepness':<22} {len(json.loads((RAW / 'footpath_steepness.geojson').read_text())['features']):,} sections")
print(f"{'GA LiDAR heights':<22} {len(json.loads((RAW / 'ga_dem_5m_samples.json').read_text())):,} points")